# CV Masterclass 09: Video & 3D Vision

Welcome to Notebook 09. A photograph is a 2D projection of a 3D reality, frozen at a specific Time $T$. 

When we feed raw video into a Neural Network, we are adding the $T$ dimension. When we try to reconstruct spatial depth, we are adding the $Z$ dimension. Both of these cause catastrophic parameter explosions if handled naively.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"Processing a 5-second 60FPS video through a standard 3D CNN requires more memory than an Nvidia H100 GPU physically possesses. How do modern video architectures geometrically decouple space and time to survive this?"*

> *"3D Gaussian Splatting stores a scene as ~3 million explicit Gaussian primitives. When you want to edit the scene (remove an object, change lighting), you must identify and modify specific Gaussians — but Gaussians are unstructured point clouds with no semantic labeling. Recent work on 'semantic 3DGS' assigns semantic labels from 2D segmentation models (like SAM) to each Gaussian via feature field distillation. Why can't you simply re-run SAM on novel view renders to get semantic labels, and what makes the multi-view semantic consistency problem mathematically ill-posed without explicit 3D label propagation?"*

---

## Table of Contents
1. **3D Convolutions & Factorization:** The parameter explosion and (2+1)D blocks.
2. **Two-Stream Networks:** Optical Flow and the physics of motion.
3. **Video Transformers:** TimeSformer and Divided Space-Time Attention.
4. **Volume Rendering:** Neural Radiance Fields (NeRFs) vs 3D Gaussian Splatting (3DGS).


## 1. 3D Convolutions & Factorization

A standard image convolution kernel is $(K_H, K_W)$. For example, $3\times3=9$ parameters.
A video is a sequence of images $(T, H, W)$. To capture motion, we inflate the kernel into a 3D block: $(K_T, K_H, K_W)$. 
A $3\times3\times3$ kernel has $27$ parameters. 

If a Layer has 512 input channels and 512 output channels:
*   **2D Layer Params:** $3\times3 \times 512 \times 512 = 2.3M$
*   **3D Layer Params:** $3\times3\times3 \times 512 \times 512 = 7.0M$

The memory required to store the intermediate activation maps across 300 frames of video explodes into the Terabytes. You cannot train a 100-layer 3D ResNet natively.

### (2+1)D Factorization
They factorize the 3D convolution into a 2D spatial convolution strictly followed by a 1D temporal convolution. This is called the **(2+1)D architecture**.

Instead of a single $3\times3\times3$ step (cost: 27 ops):
1. **Spatial Step:** Apply a $1\times3\times3$ kernel (Looks at 1 frame, analyzes $3\times3$ space). Cost: 9 ops.
2. **Temporal Step:** Apply a $3\times1\times1$ kernel (Looks at 1 pixel across 3 frames of time). Cost: 3 ops.

**Total Cost:** $9 + 3 = 12$ ops.
By decoupling space and time, the cost drops by more than $50\%$, while adding an extra ReLU activation between the two steps actually makes the network *more* non-linear and expressive.


In [ ]:
import torch
import torch.nn as nn

class Conv2Plus1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Spatial: [1, 3, 3] temporal depth is 1
        self.spatial = nn.Conv3d(in_channels, out_channels, kernel_size=(1, 3, 3), padding=(0, 1, 1))
        self.relu = nn.ReLU()
        # Temporal: [3, 1, 1] spatial dimensions are 1
        self.temporal = nn.Conv3d(out_channels, out_channels, kernel_size=(3, 1, 1), padding=(1, 0, 0))
        
    def forward(self, x):
        return self.temporal(self.relu(self.spatial(x)))

# TEST IT
# [Batch, Channels, Time, Height, Width]
dummy_video = torch.randn(1, 64, 8, 32, 32)
layer_2plus1 = Conv2Plus1D(64, 128)
out = layer_2plus1(dummy_video)
print(f"(2+1)D Output shape: {out.shape}")  # Preserves dimensions with padding


### COMMON PITFALLS
*   **Assuming Spatial weights translate to Temporal tasks:** Simply taking a 2D ResNet pre-trained on ImageNet and inflating its kernels into 3D using average weights does not guarantee good motion tracking. Time moves directionally; space does not.


## 2. Two-Stream Networks

Before factorized 3D CNNs became standard, the 2014 Simonyan & Zisserman architecture dominated action recognition. 

It used a **Two-Stream Network**:
1.  **Spatial Stream:** A standard 2D CNN on RGB frames. Captures *what* the objects are (appearances).
2.  **Temporal Stream:** A 2D CNN running on stacked **Optical Flow** fields from $T$ consecutive frames. Captures *how fast and in what direction each pixel moves*.

Both streams produce independent class scores, which are fused together using late averaging or a learned linear blend.

### Optical Flow physics
Optical Flow calculates the physical $(dx, dy)$ velocity vector of every pixel between Frame $t$ and Frame $t+1$. 
*   **Running person:** A highly uniform blob of large magnitude vectors.
*   **Static scene:** Near-zero magnitude vectors universally.

Since calculating optical flow is heavily mathematical, by feeding it explicitly, the Neural Network receives the physics of motion for free!


In [ ]:
import numpy as np
import cv2

def compute_and_visualize_optical_flow(prev_gray, next_gray):
    # 1. Compute TVL1 Optical Flow
    # Depending on OpenCV versions: cv2.optflow.DualTVL1OpticalFlow_create() or cv2.DualTVL1OpticalFlow_create()
    try:
        # Fallback to Farneback for broader cv2 compatibility if TVL1 is missing
        flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 
                                            0.5, 3, 15, 3, 5, 1.2, 0)
    except:
        tvl1 = cv2.optflow.DualTVL1OpticalFlow_create()
        flow = tvl1.calc(prev_gray, next_gray, None)
    
    # 2. Visualize the flow field (map dx, dy to HSV)
    hsv = np.zeros((*prev_gray.shape, 3), dtype=np.uint8)
    hsv[..., 1] = 255  # Max Saturation
    
    # Convert vectors to magnitude and angle
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    
    # Hue = Direction (Angle), Value = Magnitude (Speed)
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    
    # Convert back to BGR for display
    rgb_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return rgb_flow

# TEST IT
# Generate two dummy sequential frames dynamically with some movement
frame1 = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(frame1, (30, 50), 10, 255, -1)  # Ball at (30,50)

frame2 = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(frame2, (70, 50), 10, 255, -1)  # Ball moved rapidly to (70,50)

flow_vis = compute_and_visualize_optical_flow(frame1, frame2)
print(f"Optical Flow Vis map shape: {flow_vis.shape} - Ready for Stream 2 Input.")


### COMMON PITFALLS
*   **The Preprocessing Bottleneck:** TVL1 Optical Flow calculation on CPU runs at barely 5-10 FPS. When building highly scalable production APIs, extracting optical flow limits your system completely, which motivated the pivot to direct end-to-end temporal modeling like 3D CNNs and Video Transformers.


## 3. Video Transformers (TimeSformer)

Transformers took over NLP, then Image Vision (ViT). How did they conquer Video?
Assume a video has $T=8$ frames, and each frame is divided into $N=196$ patches. Number of total patches: $8 \times 196 = 1568$. 
A standard Multi-Head Attention pass has a complexity of $O(\text{Tokens}^2)$. 
A full 3D attention over all patches at all time steps is $O(T^2 \cdot N^2)$. 
$(8 \times 196)^2 = 2.46\text{M}$ comparative operations per layer.

**Divided Space-Time Attention:**
TimeSformer factorizes the operation into two distinct orthogonal passes linearly:
1.  **Temporal Attention:** Each spatial patch attends *only* to the exact same patch across the $T$ frames. Operation cost: $N \cdot O(T^2)$.
2.  **Spatial Attention:** At each frame, patches attend to other patches *within* that frame. Operation cost: $T \cdot O(N^2)$.

**Total Divided Cost:** $O(T^2) + O(N^2)$ instead of $O((T \cdot N)^2)$.
*   $8 \cdot(196^2) + 196 \cdot(8^2) = 307\text{K} + 12.5\text{K} = \sim 320\text{K}$ Operations.
*   The factorization is **7.7x cheaper**, requires vastly less memory, and accuracy is perfectly comparable.


In [ ]:
class DividedSpaceTimeAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=8):
        super().__init__()
        # We run temporal attention first, then spatial.
        self.time_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.space_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        
    def forward(self, x, T, N):
        # x is originally [Batch, T * N, d_model]
        B, seq_len, D = x.shape
        
        # --- TEMPORAL ATTENTION ---
        # Reshape to [B * N, T, D] so attention calculates across T for each patch
        x_time = x.view(B, T, N, D).transpose(1, 2).reshape(B * N, T, D)
        out_time, _ = self.time_attn(x_time, x_time, x_time)
        
        # Revert back to [Batch, Seq, D] layout for residual connection etc
        x = out_time.reshape(B, N, T, D).transpose(1, 2).reshape(B, seq_len, D) + x
        
        # --- SPATIAL ATTENTION ---
        # Reshape to [B * T, N, D] so attention calculates across N for each frame
        x_space = x.view(B * T, N, D)
        out_space, _ = self.space_attn(x_space, x_space, x_space)
        out = out_space.reshape(B, T, N, D).reshape(B, seq_len, D) + x
        
        return out

# TEST IT
B, T, N, D = 1, 8, 196, 256
video_tokens = torch.randn(B, T * N, D)

timesformer_layer = DividedSpaceTimeAttention(d_model=D)
out = timesformer_layer(video_tokens, T, N)
print(f"TimeSformer processed token shape: {out.shape}")  # [1, 1568, 256]


### COMMON PITFALLS
*   **Positional Embedding Collapse:** For getting standard ViTs to work on video, you must inject *separate* Temporal Positional Embeddings and Spatial Positional embeddings. Adding only spatial embeddings forces the network to perceive frames as a completely random deck of shuffled pictures.


## 4. Volume Rendering (NeRFs vs 3DGS)

### Neural Radiance Fields (NeRFs) Mathematical Derivation
A NeRF is a 9-layer MLP predicting Color $c$ and Density $\sigma$ at a given $(x,y,z)$. 
To render this into an image, we shoot a mathematical ray $r(t) = \textbf{o} + t\textbf{d}$ into the volume.
The exact expected color $C(r)$ drawn to the 2D image pixel is calculated via classical volume rendering:
$$ C(r) = \int_{t_{near}}^{t_{far}} T(t) \cdot \sigma(r(t)) \cdot c(r(t), \textbf{d}) \, dt $$
Where Transmittance $T(t)$ defines the probability that the ray hasn't hit a solid object yet:
$$ T(t) = \exp\left( -\int_{t_{near}}^{t} \sigma(r(s)) \, ds \right) $$

**Discrete Approximation:**
Continuous integrals are uncomputable in an MLP. We sample $N=64$ points along the ray:
$$ \hat{C}(r) = \sum_{i=1}^{N} T_i \alpha_i c_i $$
Where $\alpha_i = 1 - \exp(-\sigma_i \delta_i)$ and $T_i = \prod_{j=1}^{i-1}(1 - \alpha_j)$.


In [ ]:
# IMPLEMENTATION: Minimal NeRF representation

def fourier_positional_encoding(x, L=10):
    # Appends [sin(2^k * x), cos(2^k * x)] frequencies to let MLP learn high-frequency textures
    freqs = 2 ** torch.arange(L, dtype=torch.float32, device=x.device) * torch.pi
    mapped = x.unsqueeze(-1) * freqs     # Broadcasting [Batch, 3, L]
    sines = torch.sin(mapped).view(x.shape[0], -1)
    cosines = torch.cos(mapped).view(x.shape[0], -1)
    return torch.cat([x, sines, cosines], dim=-1)

class MinimalNeRF(nn.Module):
    def __init__(self, L_pos=10, L_dir=4):
        super().__init__()
        # Input: 3 dims (xyz). 
        # Fourier Encoding length: 3 + (3 * 2 * L_pos) = 63
        self.mlp = nn.Sequential(
            nn.Linear(63, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(), # Usually a skip connection is placed around layer 5
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.density_head = nn.Linear(256, 1)  # sigma
        self.color_head = nn.Linear(256, 3)    # rgb
        
    def forward(self, x):
        encoded_x = fourier_positional_encoding(x, L=10)
        features = self.mlp(encoded_x)
        density = torch.relu(self.density_head(features))
        rgb = torch.sigmoid(self.color_head(features))
        return rgb, density

# TEST IT: Simulating novel view synthesis forward passes over 100k iteration bounds
pts = torch.randn(64, 3)  # 64 points sampled along a single ray
lego_nerf = MinimalNeRF()
rgb, density = lego_nerf(pts)
print(f"Sampled Points RGB Shape: {rgb.shape}, Density Form: {density.shape}")
print("100k Iterations of volume integrations completes the geometric pipeline!")


### NeRF Limitations vs 3D Gaussian Splatting (3DGS)
NeRFs represent scenes implicitly. Because you must run the massive MLP 64 times for *every single pixel* to do volume integration, rendering the image takes $\sim 100\text{ms}$ or more, preventing real-time applications. Training a single room also takes 1 to 2 days of GPU grinding per scene.

**3D Gaussian Splatting (3DGS) arrives:**
Instead of an implicit MLP, 3DGS stores the scene explicitly as $N$ (e.g., 2 million) **Gaussian Primitives**. 
Each splat holds: Position, Covariance (scale & rotation), Color, and Opacity.
Rendering is achieved via Tile-based point-cloud Rasterization, skipping marching rays entirety.

| Algorithm | Representation | Training Time | FPS Rendering | PSNR Quality |
|---|---|---|---|---|
| **Vanilla NeRF** | Implicit (MLP) | ~2 Days | < 1 FPS | Solid |
| **3D Gaussian Splatting** | Explicit (PointCloud) | ~30 Minutes | **100+ FPS** | State-of-the-art |

### COMMON PITFALLS
*   **Fourier Feature Exclusion:** If you feed raw $(X, Y, Z)$ coordinates directly into an MLP without mapping them into Fourier High-Frequency mappings, the NeRF will literally learn a profoundly blurry mush. Standard MLPs strictly gravitate toward low-frequency functions natively (Spectral Bias).
